In [5]:
from pipeline_wlog import Pipeline
from utils.logging_setup import setup_latency_logger
from utils.audio_streaming import stream_audio
from utils.audio_preprocessing import preprocess_audio
from threading import Thread
import sounddevice as sd
import time
import os

In [2]:
def run_pipeline(wav_path, input_language="en", output_language="da", min_chunk_size=1000):
    audio = preprocess_audio(wav_path)

    base_name = os.path.splitext(os.path.basename(wav_path))[0]
    # build a filename to distinguish different chunk sizes
    file_name = f"{base_name}_chunk{min_chunk_size}"

    pipeline = Pipeline(input_language=input_language, output_language=output_language, min_chunk_size=min_chunk_size, file_name=file_name, device="mps")
    pipeline.start()

    def print_outputs(queue):
        while True:
            result = queue.get()
            if result is None:
                break
            transcript, translated, audio = result
            sd.play(audio, 16000)
            sd.wait()

    printer_thread = Thread(target=print_outputs, args=(pipeline.output_queue,))
    printer_thread.start()

    for i, chunk in enumerate(stream_audio(audio, frame_ms=200)):
        start_time = time.perf_counter()

        # Log the time when the audio chunk is sent to the pipeline (each sample is enumerated and time is logged) (!OBS: this is samples, not chunks)
        # But the time is only logged for the last sample of each chunk, so it will not log all samples.
        pipeline.audio_queue.put((i, start_time, chunk))

        
    pipeline.stop()
    printer_thread.join()

In [3]:
dk_data = ["data/danish/dk_speaker_1.wav", 
           "data/danish/dk_speaker_2.wav", 
           "data/danish/dk_speaker_3.wav", 
           "data/danish/dk_speaker_4.wav", 
           "data/danish/dk_speaker_5.wav",
           "data/danish/dk_speaker_6.wav",
           "data/danish/dk_speaker_7.wav",
           "data/danish/dk_speaker_8.wav",
           "data/danish/dk_speaker_9.wav",
           "data/danish/dk_speaker_10.wav"]

en_data = ["data/english/speaker_1_final.wav",
           "data/english/speaker_2_final.wav",
           "data/english/speaker_3_final.wav",
           "data/english/speaker_4_final.wav",
           "data/english/speaker_5_final.wav",
           "data/english/speaker_6_final.wav",
           "data/english/speaker_7_final.wav",
           "data/english/speaker_8_final.wav",
           "data/english/speaker_9_final.wav",
           "data/english/speaker_10_final.wav"]

chunk_sizes_testing = [3000, 3500, 4000, 4500, 5000, 5500, 6000]

--- 

## Danish to English

In [4]:
input_language = "da"
output_language = "en"

for chunk_size in chunk_sizes_testing:
    for input_file in dk_data:
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        # build a filename to distinguish different chunk sizes
        file_name = f"{base_name}_chunk{chunk_size}"

        setup_latency_logger(file_name=file_name)
        results = run_pipeline(input_file, input_language=input_language, output_language=output_language, min_chunk_size=chunk_size)
        print("logging complete")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


Loading Silero-VAD …
Loading SpeechT5 model: microsoft/speecht5_tts
Silero-VAD initialised!
Loading Danish transcriber model…


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 41630.81it/s]


SpeechT5 model loaded!


Device set to use mps:0


...

Device set to use mps:0


Danish transcriber model loaded


2025-06-14 14:04:06,211 - 15,375.97,357.95,3711.20,31820.16, i dag vil jeg tale, Today I will speak
2025-06-14 14:04:11,947 - 28,245.74,571.86,5734.55,34899.86, om energi og klima og der kan, about energy and climate and there can be
2025-06-14 14:04:12,590 - 40,109.82,604.94,643.20,33093.73, måske virke lidt overraskende for, 
2025-06-14 14:04:15,029 - 53,83.96,668.29,2438.92,32874.45, er mit fuldtidsarbejde i fonden hovedsageligt, my full-time work in the fund is mainly
2025-06-14 14:04:15,703 - 67,287.00,546.42,673.44,30700.62, er, is
2025-06-14 14:04:18,078 - 80,111.65,541.99,2374.51,30425.41, ting vi skal opfinde og levere for, thing we must invent and deliver for
2025-06-14 14:04:20,199 - 92,91.72,625.76,2120.71,30089.40, at hjælpe de fattigste milliarder mennesker, to help the poorest billion people
2025-06-14 14:04:22,519 - 105,108.89,606.57,2320.03,29760.20, til et bedre liv men energi og, for a better life but energy and
2025-06-14 14:04:24,930 - 118,74.15,622.02,2411.00,2950

KeyboardInterrupt: 

2025-06-14 14:05:03,786 - 365,126.05,504.40,3503.22,18021.62, avanceret civilisation er, advanced civilization is
2025-06-14 14:05:05,552 - 379,223.75,524.48,1765.90,16937.53, baseret på fremskridt inden for energi, based on progress in energy
2025-06-14 14:05:07,505 - 392,262.19,476.89,1952.83,16247.29, kold revolutionen drev den inde, cold revolution drives it inside
2025-06-14 14:05:09,385 - 407,320.46,493.28,1879.75,15074.95, revolution og, revolution and
2025-06-14 14:05:11,420 - 422,247.44,486.10,2034.62,14062.71, i 1900 tallet set et, in the 1900s seen a
2025-06-14 14:05:13,384 - 435,123.62,584.75,1963.97,13387.24, hurtigt fald i prisen på, rapid drop in the price of


### Evaluating:

---

## English to Danish

In [ ]:
input_language = "en"
output_language = "da"

for chunk_size in chunk_sizes_testing:
    for input_file in en_data:
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        # build a filename to distinguish different chunk sizes
        file_name = f"{base_name}_chunk{chunk_size}"
        setup_latency_logger(file_name=f"{file_name}")

        full_transcription, full_translation = run_pipeline(input_file, input_language=input_language, output_language=output_language, min_chunk_size=chunk_size)
        
        # write the results to separate text files
        with open(f"results/{file_name}_transcription.txt", "w", encoding="utf-8") as f_trans:
            f_trans.write(full_transcription)

        with open(f"results/{file_name}_translation.txt", "w", encoding="utf-8") as f_transl:
            f_transl.write(full_translation)